# Frozen G3C R4 official 1,012 — embedding phase

This is a post-freeze engineering run, not a public selection run. It sends intact frozen embedding batches to two matching T4 GPUs and requires an exact numeric Promotion canary on both devices.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
matches = sorted(Path('/kaggle/input').rglob('g3c_official_payload_manifest.json'))
assert len(matches) == 1, f'expected one official payload, found {matches}'
PAYLOAD = matches[0].parent
manifest = json.loads(matches[0].read_text(encoding='utf-8'))
assert manifest['mode'] == 'official_engineering_audit'
assert manifest['question_count'] == 1012 and manifest['selected_stage'] == 'R4'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PAYLOAD / manifest['paths']['requirements'])], check=True)
print({'payload': str(PAYLOAD), 'fingerprint': manifest['payload_fingerprint'], 'questions': manifest['question_count']})

In [ ]:
os.environ['HF_HOME'] = '/kaggle/temp/g3c_official_hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
sys.dont_write_bytecode = True
sys.path.insert(0, str(PAYLOAD / 'code'))
import torch, transformers
from vifinqa.g3c_official.payload import validate_official_payload
assert transformers.__version__ == '4.53.3', transformers.__version__
assert torch.cuda.is_available() and torch.cuda.device_count() == 2
gpu = [(torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i), torch.cuda.get_device_properties(i).total_memory) for i in range(2)]
assert gpu[0] == gpu[1], gpu
validated = validate_official_payload(PAYLOAD)
assert validated['payload_fingerprint'] == manifest['payload_fingerprint']
print({'gpus': gpu, 'torch': torch.__version__, 'transformers': transformers.__version__, 'payload_valid': True})

In [ ]:
OUT = Path('/kaggle/working/g3c_official_embedding')
runner = PAYLOAD / manifest['paths']['runner']
command = [sys.executable, str(runner), 'embedding', '--payload', str(PAYLOAD), '--out', str(OUT), '--backend', 'qwen']
print('Starting exact two-T4 embedding phase')
subprocess.run(command, check=True)

In [ ]:
from vifinqa.g3c_official.execution import validate_embedding_result
report = validate_embedding_result(payload_dir=PAYLOAD, result_dir=OUT, require_qwen=True)
assert report['exact_canary_passed_on_both_gpus'] is True
print(json.dumps({'run_signature': report['run_signature'], 'vectors': report['vector_count'], 'seconds': report['total_seconds']}, indent=2))

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/g3c_official_embedding_results', 'zip', root_dir=OUT)
print({'save_version_directory': str(OUT), 'download_zip': archive})